# Hälsostudie: Del 1

Rapporten beskriver deltagarnas uppmätta hälsomarkörer och undersöker 4 frågor:

1. Hur fördelar sig hälsomåtten och hur stor andel av deltagarna röker?
2. Hur skiljer sig den observerade sjukdomsandelen från andelen bland 1000 simulerade personer?
3. Vilken är den statistiska osäkerheten kring det uppskattade medelblodtrycket?
4. Har rökare högre systoliskt medelblodtryck än icke-rökare, och hur ofta kan man tänka sig att samma test skulle upptäcka en liknande skillnad?


## Analysinställningar

Konfidensgraden är 95 % och signifikansnivån är 5 %. Simuleringen, bootstrap och power-simuleringen använder varsin slumptalsgenerator med ett förbestämt seed.


In [1]:
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from scipy import stats

confidence = 0.95
alpha = 0.05
disease_seed = 45
bootstrap_seed = 46
power_seed = 47

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (7.2, 3.7),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelcolor": "#243649",
    "grid.alpha": 0.24,
    "axes.axisbelow": True,
})
blue = "#276487"
teal = "#147D73"
ochre = "#AC6327"

print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · NumPy {np.__version__}")
print(f"Matplotlib {matplotlib.__version__} · SciPy {scipy.__version__}")


Python 3.14.7 · pandas 3.1.1 · NumPy 2.5.4
Matplotlib 3.11.2 · SciPy 1.18.1


## Data och kvalitetskontroll

Varje rad motsvarar en deltagare. ID används för att kontrollera radens identitet. Sjukdom är kodad som 1 för registrerad sjukdom och 0 för övriga deltagare. Rökstatus är kodad som Yes eller No.

Kontrollen omfattar kolumner, ID, dubbletter, numeriska värden, tillåtna kategorier och saknade fält. Varje beräkning använder de deltagare som har giltiga värden för just de variabler som behövs.


In [2]:
csv_path = Path("data/health_study_dataset.csv")
data_path = next(
    (path for path in (csv_path, Path("..") / csv_path) if path.is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError("Kör notebooken från projektroten eller mappen notebooks.")

df = pd.read_csv(data_path, encoding="utf-8")
measurements = {
    "age": ("Ålder", "år"),
    "weight": ("Vikt", "kg"),
    "height": ("Längd", "cm"),
    "systolic_bp": ("Systoliskt blodtryck", "mmHg"),
    "cholesterol": ("Kolesterol", "mmol/L"),
}
numeric_columns = [*measurements, "disease"]
expected_columns = {"id", "sex", "smoker", *numeric_columns}
if set(df.columns) != expected_columns:
    raise ValueError("CSV-filens kolumner avviker från det förväntade schemat.")
if df.empty:
    raise ValueError("CSV-filen innehåller inga deltagare.")

duplicate_rows = int(df.duplicated().sum())
unique_ids = df["id"].nunique()
if duplicate_rows or df["id"].isna().any() or unique_ids != len(df):
    raise ValueError("Varje rad ska ha ett unikt deltagar-ID.")

df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric)
if not all(np.isfinite(df[column].dropna()).all() for column in numeric_columns):
    raise ValueError("En numerisk kolumn innehåller oändliga värden.")

categories = {"sex": {"F", "M"}, "smoker": {"Yes", "No"}, "disease": {0, 1}}
for column, allowed in categories.items():
    if not set(df[column].dropna()).issubset(allowed):
        raise ValueError(f"Otillåtet kategorivärde i {column}.")

print(f"Deltagare {len(df)} | Unika ID {unique_ids} | Identiska dubbletter {duplicate_rows}")
display(df.isna().sum().rename("Saknade värden").to_frame())


Deltagare 800 · Unika ID 800 · Identiska dubbletter 0


Alla 800 deltagare har unika ID och fullständiga uppgifter i filen. Samtliga rader ingår i analysen. Uppgifter om urvalsmetod och eventuellt beroende mellan deltagarna saknas i uppgiftsunderlaget, vilket begränsar hur långt resultaten kan generaliseras.


## Beskrivande statistik

Tabellen sammanfattar de fem hälsomåtten. Medelvärde och median visar var värdena ligger i mitten av fördelningen. Min och max visar det observerade spannet.


In [3]:
summary = df[list(measurements)].agg(["count", "mean", "median", "min", "max"]).T
summary.columns = ["Antal", "Medelvärde", "Median", "Min", "Max"]
summary["Antal"] = summary["Antal"].astype(int)
summary.insert(0, "Enhet", [unit for _, unit in measurements.values()])
summary.index = pd.Index([name for name, _ in measurements.values()], name="Mått")
display(summary.round(2))


### Systoliskt blodtryck


In [4]:
bp = df["systolic_bp"].dropna().to_numpy()
bp_mean = bp.mean()
bp_bins = np.histogram_bin_edges(bp, bins="auto")

fig, ax = plt.subplots(layout="constrained")
ax.hist(bp, bins=bp_bins, color=blue, edgecolor="white")
ax.axvline(bp_mean, color=ochre, linestyle="--", linewidth=2, label="Medelvärde")
ax.set(title="Fördelning av systoliskt blodtryck",
       xlabel="Systoliskt blodtryck (mmHg)", ylabel="Antal deltagare")
ax.grid(axis="y")
ax.legend(frameon=False)
plt.show()


Fördelningen är ungefär symmetrisk kring 149 mmHg. Den mittersta hälften ligger mellan 140,9 och 157,6 mmHg. Det lägsta och högsta registrerade värdet är 106,8 respektive 185,9 mmHg.

### Vikt efter kön


In [5]:
weight_groups = [
    df.loc[df["sex"].eq(category), "weight"].dropna()
    for category in ("F", "M")
]

fig, ax = plt.subplots(layout="constrained")
ax.boxplot(
    weight_groups,
    tick_labels=["Kvinnor (F)", "Män (M)"],
    widths=0.45,
    patch_artist=True,
    boxprops={"facecolor": "#DAEAF3", "edgecolor": blue},
    medianprops={"color": blue, "linewidth": 2},
    flierprops={"marker": "o", "markersize": 3, "alpha": 0.55},
)
ax.set(title="Viktfördelning efter kön", xlabel="Kön", ylabel="Vikt (kg)")
ax.grid(axis="y")
plt.show()


Medianvikten är 67,4 kg i grupp F och 78,9 kg i grupp M. Boxarna visar att gruppernas mittersta hälfter överlappar trots skillnaden i median.

### Rökvanor


In [6]:
smoking_counts = df["smoker"].value_counts().reindex(["No", "Yes"], fill_value=0)
smoking_shares = smoking_counts / smoking_counts.sum()
labels = [
    f"{share:.1%} ({count})".replace(".", ",")
    for share, count in zip(smoking_shares, smoking_counts)
]

fig, ax = plt.subplots(layout="constrained")
bars = ax.bar(["Icke-rökare", "Rökare"], smoking_shares, color=[blue, teal], width=0.5)
ax.bar_label(bars, labels=labels, padding=5)
ax.set(title="Andel deltagare efter rökstatus", ylabel="Andel deltagare", ylim=(0, 1))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(axis="y")
plt.show()


Av 800 deltagare är 213 (26,6 %) rökare och 587 (73,4 %) icke-rökare. De observerade gruppstorlekarna används i hypotesprövningen och power-simuleringen.


## Simulering av sjukdomsförekomst

Sannolikheten i simuleringen sätts till andelen deltagare med sjukdomen i filen. Varje simulerad person får ett oberoende binärt utfall med denna sannolikhet. Ett binomialdrag med ett försök ger just ett sådant utfall. [(Se NumPy-dokumentationen här)](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.binomial.html)


In [7]:
disease = df["disease"].dropna().to_numpy()
observed_share = disease.mean()
disease_rng = np.random.default_rng(disease_seed)
simulated_disease = disease_rng.binomial(n=1, p=observed_share, size=1_000)
simulated_share = simulated_disease.mean()
difference_pp = 100 * (simulated_share - observed_share)

prevalence = pd.DataFrame(
    {
        "Antal personer": [len(disease), len(simulated_disease)],
        "Antal med sjukdom": [int(disease.sum()), int(simulated_disease.sum())],
        "Andel (%)": [100 * observed_share, 100 * simulated_share],
    },
    index=["Observerat", "Simulerat"],
)
display(prevalence.round(3))
difference_label = f"{difference_pp:+.3f}".replace(".", ",")
print(f"Simulerad andel minus observerad andel är {difference_label} procentenheter.")


Simulerad andel minus observerad andel är -0,775 procentenheter.


I filen har 47 av 800 deltagare sjukdomen (5,875 %). Simuleringen gav 51 av 1000 (5,1 %), en skillnad på −0,775 procentenheter. Skillnaden illustrerar slumpvariationen när varje simulerad person har samma sannolikhet för sjukdom.


## Konfidensintervall för medelblodtrycket

Båda metoderna skattar osäkerheten kring samma populationsmedelvärde under jämförbara urvalsantaganden. Normalapproximationen använder stickprovets standardavvikelse $s$ och standardfelet $s/\sqrt{n}$. Med 800 observationer och en ungefär symmetrisk fördelning är en normalapproximation av stickprovsmedelvärdet ett rimligt val, under antagandet om tillräckligt oberoende observationer.

Percentilbootstrap uppskattar medelvärdets stickprovsfördelning genom att dra 10 000 nya stickprov med återläggning. Varje stickprov har samma storlek som originalet. Den 2,5:e och 97,5:e percentilen av de nya medelvärdena blir intervallgränser. Jämförelsen visar hur mycket valet av metod påverkar skattningen av osäkerheten. [Se: ISLP, 3.1.2 och 5.2](https://www.statlearning.com/)


In [8]:
bp_sd = bp.std(ddof=1)
bp_se = bp_sd / np.sqrt(bp.size)
tail = (1 - confidence) / 2
quantiles = [tail, 1 - tail]
normal_ci = bp_mean + stats.norm.ppf(quantiles) * bp_se

bootstrap_rng = np.random.default_rng(bootstrap_seed)
bootstrap_repetitions = 10_000
bootstrap_means = np.array([
    bootstrap_rng.choice(bp, size=bp.size, replace=True).mean()
    for _ in range(bootstrap_repetitions)
])
bootstrap_ci = np.quantile(bootstrap_means, quantiles)

intervals = pd.DataFrame(
    [normal_ci, bootstrap_ci],
    index=["Normalapproximation", "Percentilbootstrap"],
    columns=["Nedre gräns (mmHg)", "Övre gräns (mmHg)"],
)
intervals.insert(0, "Medelvärde (mmHg)", bp_mean)
intervals["Bredd (mmHg)"] = intervals["Övre gräns (mmHg)"] - intervals["Nedre gräns (mmHg)"]

display(pd.DataFrame({
    "Antal": [bp.size],
    "Standardavvikelse (mmHg)": [bp_sd],
    "Standardfel (mmHg)": [bp_se],
}).round(3))
display(intervals.round(3))


Det skattade medelvärdet är 149,179 mmHg. Normalapproximationens 95-procentiga intervall är 148,292–150,065 mmHg och bootstrapintervallet är 148,279–150,049 mmHg. Intervallen har nästan samma bredd, 1,773 respektive 1,770 mmHg, och gränserna skiljer sig med mindre än 0,02 mmHg. Här ger metoderna därför en liknande bild av precisionen.

Vid upprepade jämförbara stickprov skulle ungefär 95 % av intervallen från respektive metod täcka populationens medelvärde om metodens antaganden håller. Precisionen i intervallen säger i sig lite om hur representativt urvalet är.


## Hypotesprövning av rökning och blodtryck

Låt $\Delta$ beteckna rökarnas populationsmedelvärde minus icke-rökarnas. Uppgiftens riktade fråga ger $H_0\colon \Delta \leq 0$ och $H_1\colon \Delta > 0$. Testet beräknas vid nollhypotesens gräns, $\Delta = 0$.

Welchs ensidiga t-test passar jämförelsen av två oberoende gruppmedelvärden och tillåter olika gruppvarianser. Diagrammen använder gemensamma intervallgränser och täthet för att kunna jämföra fördelningarnas form trots att grupperna är olika stora. Oberoende inom och mellan grupperna är ett antagande för testet. [SciPy-dokumentation för Welchs t-test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html)


In [9]:
smokers_bp = df.loc[df["smoker"].eq("Yes"), "systolic_bp"].dropna().to_numpy()
nonsmokers_bp = df.loc[df["smoker"].eq("No"), "systolic_bp"].dropna().to_numpy()
bp_groups = [smokers_bp, nonsmokers_bp]
group_names = ["Rökare", "Icke-rökare"]
group_means = [sample.mean() for sample in bp_groups]
group_sds = [sample.std(ddof=1) for sample in bp_groups]

fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(8.8, 3.2), layout="constrained")
for ax, sample, name, color in zip(axes, bp_groups, group_names, [teal, blue]):
    ax.hist(sample, bins=bp_bins, density=True, color=color, edgecolor="white")
    ax.set(title=f"{name} (n={sample.size})", xlabel="Systoliskt blodtryck (mmHg)")
    ax.grid(axis="y")
axes[0].set_ylabel("Täthet (1/mmHg)")
plt.show()


def welch_greater(smokers, nonsmokers):
    return stats.ttest_ind(
        smokers, nonsmokers, equal_var=False, alternative="greater", axis=-1, nan_policy="raise"
    )


group_summary = pd.DataFrame(
    {
        "Antal": [sample.size for sample in bp_groups],
        "Medelvärde (mmHg)": group_means,
        "Standardavvikelse (mmHg)": group_sds,
    },
    index=group_names,
)
observed_test = welch_greater(smokers_bp, nonsmokers_bp)
mean_difference = group_means[0] - group_means[1]

display(group_summary.round(3))
display(pd.DataFrame({
    "Skillnad i medelvärde (mmHg)": [mean_difference],
    "t-värde": [observed_test.statistic],
    "p-värde (ensidigt)": [observed_test.pvalue],
    "Signifikansnivå": [alpha],
}).round(4))


Blodtrycksfördelningarna har liknande form och överlappar till stor del. Rökarnas medelvärde är 149,52 mmHg och icke-rökarnas 149,05 mmHg, en observerad skillnad på 0,47 mmHg. Det ensidiga testet ger p = 0,3264. Vid signifikansnivån 0,05 kan vi därför inte förkasta nollhypotesen. Under testets antaganden ger materialet otillräckligt stöd för ett högre systoliskt medelblodtryck bland rökare. Jämförelsen är ojusterad och kan påverkas av andra gruppskillnader, exempelvis ålder.


## Testets statistiska styrka

Statistisk styrka är sannolikheten att förkasta nollhypotesen vid en viss sann skillnad. För varje scenario simuleras 5000 nya studier med 213 rökare och 587 icke-rökare. Båda grupperna simuleras som oberoende normalfördelade observationer med varsin standardavvikelse skattad från filen. De observerade fördelningarna är ungefär symmetriska, vilket ger stöd för modellen som en approximation.

Icke-rökarnas observerade medelvärde används som gemensam basnivå. Rökarnas simulerade medelvärde höjs med 0, 2, 5 eller 10 mmHg för att illustrera testets beteende över ett spann av skillnader. Vid 0 mmHg uppskattas risken för typ I-fel vid nollhypotesens gräns. Samma ensidiga Welch-test och signifikansnivå används i alla scenarier. [ISLP, 13.1.2](https://www.statlearning.com/); [SciPys beskrivning av simulerad styrka](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.power.html)


In [11]:
power_rng = np.random.default_rng(power_seed)
power_repetitions = 5_000
scenarios = [0, 2, 5, 10]
baseline = group_means[1]
power_rows = []

for delta in scenarios:
    simulated_smokers = power_rng.normal(
        baseline + delta,
        group_sds[0],
        size=(power_repetitions, smokers_bp.size)
    )
    simulated_nonsmokers = power_rng.normal(
        baseline,
        group_sds[1],
        size=(power_repetitions, nonsmokers_bp.size)
    )
    simulation_test = welch_greater(simulated_smokers, simulated_nonsmokers)
    rejections = int(np.count_nonzero(simulation_test.pvalue < alpha))
    rate = rejections / power_repetitions
    mc_se = np.sqrt(rate * (1 - rate) / power_repetitions)
    power_rows.append((rejections, 100 * rate, 100 * mc_se))

power_table = pd.DataFrame(
    power_rows,
    index=pd.Index(scenarios, name="Simulerad skillnad (mmHg)"),
    columns=[
        "Antal förkastanden",
        "Andel förkastanden (%)",
        "Monte Carlo-SE (procentenheter)"
    ],
)
power_table.insert(0, "Antal studier", power_repetitions)
display(power_table.round(3))

rejection_rates = power_table["Andel förkastanden (%)"] / 100
fig, ax = plt.subplots(layout="constrained")
ax.plot(scenarios[1:], rejection_rates.iloc[1:], "o-", color=blue, label="Simulerad styrka")
ax.scatter(0, rejection_rates.loc[0], color=ochre, zorder=3, label="Förkastanden vid nollskillnad")
ax.axhline(alpha, color="#66788A", linestyle="--", label="Signifikansnivå 5 %")
ax.set(
    title="Welch-testet, förkastanden av nollhypotesen",
    xlabel="Antagen skillnad (mmHg)",
    ylabel="Andel test som förkastar nollhypotesen",
    xticks=scenarios,
    yticks=np.linspace(0, 1, 6),
    ylim=(-0.025, 1.055)
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(axis="y")
ax.legend(loc="center right", frameon=False)
plt.show()


Vid nollskillnad förkastades nollhypotesen i 4,8 % av studierna, nära signifikansnivån 5 %. Vid en antagen skillnad på 2 mmHg var den uppskattade styrkan 60,9 % och vid 5 mmHg 99,92 %. Vid 10 mmHg förkastades nollhypotesen i samtliga 5000 simulerade studier.

Monte Carlo-standardfelet visar hur mycket andelen kan variera mellan nya serier av samma simulering. Vid 2 mmHg är det 0,69 procentenheter. Vid 10 mmHg ger formeln noll eftersom alla simulerade test förkastade nollhypotesen. Det utfallet bygger fortfarande på ett ändligt antal simuleringar. Styrkeskattningarna gäller de angivna gruppstorlekarna, standardavvikelserna och modellantagandena.


## Slutsatser

Det systoliska blodtrycket låg i genomsnitt på 149,18 mmHg och de två metoderna gav nästan identiska konfidensintervall för medelvärdet. Sjukdomssimuleringen gav 5,1 % jämfört med 5,875 % i filen, vilket illustrerar variationen i en simulerad andel. Rökarnas observerade medelblodtryck var 0,47 mmHg högre, men det ensidiga testet gav otillräckligt statistiskt stöd för hypotesen. Styrkesimuleringen visar att den här jämförelsen kan missa en skillnad på 2 mmHg relativt ofta under de valda antagandena.

Resultaten gäller den analyserade filen. Tolkning på populationsnivå förutsätter ett lämpligt urval och tillräckligt oberoende observationer. En bedömning av en eventuell orsakseffekt skulle kräva mer information om studiedesignen och möjliga förväxlingsfaktorer.

## Källor

- James, G., Witten, D., Hastie, T., Tibshirani, R. & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. Avsnitt 2.3 för data och grafik, 3.1.2 för standardfel och konfidensintervall, 5.2 för bootstrap och 13.1 för hypotesprövning och styrka. [Hämtat från bokens webbplats](https://www.statlearning.com/)
- NumPy. [Slumptalsgenerator med fast seed](https://numpy.org/doc/stable/reference/random/generator.html) och [binomialfördelning](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.binomial.html).
- SciPy. [Oberoende t-test med Welch-alternativet](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html) och [simulerad statistisk styrka](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.power.html).
